# Notebook 1 — Where does each PEFT method operate?

This notebook is a **minimal, visual PEFT anatomy guide with zoom-ins** for a handful of PEFT families.

We use a frozen **vision backbone** (a tiny ViT) and ask:

- **Visual prompt tuning:** how do trainable patches modify the **input image space**?
- **Adapters:** how do residual bottlenecks modify the **hidden-state / activation path**?
- **LoRA:** how do low-rank matrices modify the **weight space** of specific linear layers?
- **BitFit:** what happens when we touch **only bias parameters**?
- **Linear probing:** what if the encoder is frozen and only the **readout head** changes?
- **Partial fine-tuning:** what if we unfreeze the **last transformer block**?

All PEFT wrapper classes are imported from `src/methods/`. The notebook focuses on intuition and visualisation.

In [ ]:
# Optional install cell for Colab / fresh environments
# !pip install -q torch torchvision matplotlib pandas peft

In [ ]:
import sys
import copy
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle, FancyArrowPatch

ROOT = Path.cwd()
if (ROOT / ".." / "src").exists():
    sys.path.insert(0, str((ROOT / "..").resolve()))
elif (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT.resolve()))

from src.methods.linear_probe import LinearProbeModel
from src.methods.adapters import AdapterHeadClassifier
from src.methods.prompt_tuning import PromptTunedClassifier
from src.methods.lora import LoRAClassifier
from src.methods.bitfit import BitFitClassifier
from src.methods.partial_ft import PartialFineTuneClassifier
from src.training import count_trainable_parameters, freeze_module

torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 1) A frozen vision backbone

We define a **TinyViT**: a minimal Vision Transformer with 2 transformer layers and embedding dimension 128.
It takes an image `[B, 3, 32, 32]`, splits it into 4x4 patches, processes them through a transformer,
and returns the CLS token as a `[B, 128]` feature vector.

We **freeze all parameters** of this backbone. Every PEFT method must adapt it without touching these frozen weights directly.

In [ ]:
class TinyViT(nn.Module):
    """
    Minimal Vision Transformer backbone.
    Input : [B, 3, img_size, img_size]
    Output: [B, embed_dim]  -- CLS token after final LayerNorm
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=128, depth=2, num_heads=4):
        super().__init__()
        self.patch_embed = nn.Conv2d(
            in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        n_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed  = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 2, dropout=0.0, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.feature_dim = embed_dim

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)  # [B, n_patches, D]
        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)                    # [B, n_patches+1, D]
        x   = x + self.pos_embed
        x   = self.transformer(x)
        return self.norm(x)[:, 0]                           # CLS token: [B, D]


backbone = TinyViT().to(device)
freeze_module(backbone)

total = sum(p.numel() for p in backbone.parameters())
print(f"TinyViT total parameters : {total:,}")
print(f"Feature dimension        : {backbone.feature_dim}")


## 2) A shared synthetic image batch

We create one mini-batch of random images to probe each method before any training.

In [ ]:
B, C, H, W = 8, 3, 32, 32
x = torch.randn(B, C, H, W, device=device)

with torch.no_grad():
    h_base = backbone(x)

print("Input  shape:", x.shape)
print("Backbone output shape:", h_base.shape)


## 3) PEFT variants — imported from `src/methods`

Each wrapper below isolates **where** the trainable change enters the computation.
No class definitions here: the adaptation logic lives in `src/methods/`.

| Method | Class | What is trainable |
|---|---|---|
| Linear probing | `LinearProbeModel` | classifier head only |
| Visual prompt tuning | `PromptTunedClassifier` | learnable image-space patch |
| Adapter | `AdapterHeadClassifier` | bottleneck branch after backbone |
| LoRA | `LoRAClassifier` | A and B matrices on FFN layers (via PEFT) |
| BitFit | `BitFitClassifier` | backbone bias parameters + head |
| Partial fine-tuning | `PartialFineTuneClassifier` | last transformer block + head |

All backbones are **deep copies** of the same frozen checkpoint so comparisons are meaningful.

In [ ]:
N_CLASSES = 3
D = backbone.feature_dim

# Partial-FT: extract last block reference before the constructor freezes everything
bb_pf      = copy.deepcopy(backbone)
last_block = list(bb_pf.transformer.layers)[-1]

variants = {
    "linear_probe" : LinearProbeModel(
        copy.deepcopy(backbone), D, N_CLASSES),
    "visual_prompt": PromptTunedClassifier(
        copy.deepcopy(backbone), D, N_CLASSES, prompt_size=8),
    "adapter"      : AdapterHeadClassifier(
        copy.deepcopy(backbone), D, N_CLASSES, bottleneck_dim=32),
    "lora"         : LoRAClassifier(
        copy.deepcopy(backbone), D, N_CLASSES,
        target_modules=["linear1", "linear2"], rank=8),
    "bitfit"       : BitFitClassifier(
        copy.deepcopy(backbone), D, N_CLASSES),
    "partial_ft"   : PartialFineTuneClassifier(
        bb_pf, D, N_CLASSES, modules_to_unfreeze=[last_block]),
}

pd.DataFrame(
    [{"method": name, "trainable_params": count_trainable_parameters(model)}
     for name, model in variants.items()]
).sort_values("trainable_params").reset_index(drop=True)


## 4) Which PEFT method shifts the backbone features?

We compare each variant's **backbone output** to the frozen reference.

A non-zero shift means the method changes **what the backbone produces** --
either by modifying the input it sees (visual prompt) or by unfreezing some of its layers (partial fine-tuning).

Methods like adapters and LoRA leave backbone features unchanged at initialisation;
they only modify the **downstream** computation.

In [ ]:
def rep_stats(h_ref, h_new):
    rel_l2 = (h_new - h_ref).norm(dim=-1).mean() / (h_ref.norm(dim=-1).mean() + 1e-8)
    cos = F.cosine_similarity(h_ref, h_new, dim=-1).mean()
    return float(rel_l2.detach().cpu()), float(cos.detach().cpu())


def get_backbone_features(name, model, x):
    with torch.no_grad():
        if name == "visual_prompt":
            return model.backbone(model.prompt(x))
        return model.backbone(x)


rows = []
with torch.no_grad():
    h_ref = backbone(x)

for name, model in variants.items():
    h_new = get_backbone_features(name, model, x)
    rel_l2, cos = rep_stats(h_ref, h_new)
    rows.append({
        "method": name,
        "relative_L2_shift": round(rel_l2, 6),
        "mean_cosine_to_base": round(cos, 4),
        "changes_backbone_features": "yes" if rel_l2 > 1e-6 else "no",
    })

df_shift = pd.DataFrame(rows).sort_values("relative_L2_shift")
df_shift


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_shift["method"], df_shift["relative_L2_shift"])
plt.ylabel("Relative L2 shift in backbone features")
plt.title("Which PEFT method changes what the backbone produces?")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 5) Visual zoom: where does the PEFT state live?

The next cells are teaching diagrams that zoom from the whole model into one specific component.

- **Visual prompt tuning** adds trainable patches before the frozen model.
- **Adapters** add a small trainable residual branch inside the hidden-state path.
- **LoRA** keeps the pretrained weight frozen but adds $W_{eff}=W_0 + BA$.
- **BitFit** updates small existing parameter subsets.
- **Linear probing** trains only the readout head.
- **Partial fine-tuning** unfreezes selected pretrained blocks.

In [ ]:
def add_box(ax, xy, w, h, text, fc="white", lw=1.5, fontsize=10):
    box = Rectangle(xy, w, h, facecolor=fc, edgecolor="black", linewidth=lw)
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box


def add_arrow(ax, start, end, text=None, rad=0.0):
    arrow = FancyArrowPatch(
        start, end, arrowstyle="->", mutation_scale=13, linewidth=1.3,
        connectionstyle=f"arc3,rad={rad}", color="black")
    ax.add_patch(arrow)
    if text:
        ax.text((start[0]+end[0])/2, (start[1]+end[1])/2 + 0.18, text, ha="center", fontsize=9)


def draw_peft_map():
    fig, ax = plt.subplots(figsize=(12, 5.5))
    ax.set_xlim(0, 12); ax.set_ylim(0, 7); ax.axis("off")
    add_box(ax, (0.5, 3.0), 1.5, 0.8, "input\nimage")
    add_box(ax, (2.7, 3.0), 1.7, 0.8, "patch\nembedding")
    add_box(ax, (5.0, 3.0), 2.2, 0.8, "frozen\nTransformer blocks")
    add_box(ax, (8.0, 3.0), 1.7, 0.8, "CLS\nfeatures")
    add_box(ax, (10.3, 3.0), 1.2, 0.8, "task\nhead")
    for s, e in [((2.0,3.4),(2.7,3.4)),((4.4,3.4),(5.0,3.4)),((7.2,3.4),(8.0,3.4)),((9.7,3.4),(10.3,3.4))]:
        add_arrow(ax, s, e)
    add_box(ax, (2.2, 5.3), 2.3, 0.8, "visual prompt\ntrainable patch", fc="#f3f3f3")
    add_arrow(ax, (3.35, 5.3), (3.35, 3.85), "perturb input")
    add_box(ax, (4.9, 1.2), 2.4, 0.8, "adapter\nbottleneck branch", fc="#f3f3f3")
    add_arrow(ax, (6.1, 3.0), (6.1, 2.0), "hidden-state residual")
    add_box(ax, (5.1, 5.3), 2.0, 0.8, "LoRA\ndW = BA", fc="#f3f3f3")
    add_arrow(ax, (6.1, 5.3), (6.1, 3.85), "weight update")
    add_box(ax, (7.6, 5.3), 1.8, 0.8, "BitFit\nsmall params", fc="#f3f3f3")
    add_arrow(ax, (8.5, 5.3), (7.0, 3.85), "parameter subset", rad=-0.2)
    add_box(ax, (9.9, 1.2), 1.8, 0.8, "linear probe\nhead only", fc="#f3f3f3")
    add_arrow(ax, (10.8, 2.0), (10.9, 3.0), "readout only")
    ax.text(6, 6.65, "Same frozen checkpoint, different intervention sites",
            ha="center", fontsize=14, fontweight="bold")
    plt.show()


draw_peft_map()


### Zoom-in: LoRA inside one frozen linear projection

LoRA trains two smaller matrices instead of updating $W_0$ directly:
- $A \in \mathbb{R}^{r \times d_{in}}$ projects down to rank $r$
- $B \in \mathbb{R}^{d_{out} \times r}$ projects back up
- effective layer: $x(W_0 + BA)^T$

In our workshop we apply this to the FFN `linear1` / `linear2` layers inside each ViT block via the PEFT library.

In [ ]:
def draw_lora_zoom(d_in=12, d_out=10, rank=3):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.set_xlim(0, 14); ax.set_ylim(0, 6); ax.axis("off")
    ax.text(7, 5.65, "LoRA zoom-in: frozen weight + low-rank trainable update",
            ha="center", fontsize=14, fontweight="bold")
    add_box(ax, (0.5, 2.6), 1.1, 0.8, "x")
    add_box(ax, (12.4, 2.6), 1.1, 0.8, "y")
    add_arrow(ax, (1.6, 3.0), (2.3, 3.0))
    add_arrow(ax, (11.7, 3.0), (12.4, 3.0))
    add_box(ax, (2.3, 2.1), 2.2, 1.8, f"frozen W0\n{d_out}x{d_in}", fc="#ffffff", lw=2)
    add_arrow(ax, (4.5, 3.0), (5.4, 3.0), "base path")
    add_box(ax, (2.3, 0.4), 1.6, 0.9, f"A\n{rank}x{d_in}", fc="#f3f3f3")
    add_box(ax, (4.5, 0.4), 1.6, 0.9, f"B\n{d_out}x{rank}", fc="#f3f3f3")
    add_arrow(ax, (1.05, 2.6), (3.1, 1.3), "parallel low-rank path", rad=0.25)
    add_arrow(ax, (3.9, 0.85), (4.5, 0.85))
    add_arrow(ax, (6.1, 0.85), (7.0, 2.45), "dW = BA", rad=-0.2)
    add_box(ax, (7.0, 2.45), 1.1, 1.1, "+")
    add_arrow(ax, (8.1, 3.0), (9.0, 3.0))
    add_box(ax, (9.0, 2.2), 2.7, 1.6, "effective layer\nW_eff = W0 + BA")
    full = d_in * d_out; lora = rank * d_in + d_out * rank
    ax.text(7, 1.55, f"Full update trains {full} numbers. LoRA trains {lora} at rank r={rank}.",
            ha="center", fontsize=11)
    ax.text(7, 1.15, "Applied via PEFT library to FFN linear layers inside each ViT block.",
            ha="center", fontsize=10)
    plt.show()


draw_lora_zoom()


In [ ]:
torch.manual_seed(3)
d_in, d_out, rank = 16, 16, 3
W0 = torch.randn(d_out, d_in)
A  = torch.randn(rank, d_in) * 0.05
B  = torch.randn(d_out, rank) * 0.05
delta_W = B @ A
W_eff   = W0 + delta_W

print("Frozen W0 shape:  ", tuple(W0.shape))
print("LoRA A shape:     ", tuple(A.shape))
print("LoRA B shape:     ", tuple(B.shape))
print("Rank(Delta W):    ", int(torch.linalg.matrix_rank(delta_W)))
print("Trainable params: ", A.numel() + B.numel(), "vs full matrix:", W0.numel())

for title, mat in [("Frozen weight W0", W0), ("LoRA update dW=BA", delta_W), ("Effective W0+dW", W_eff)]:
    plt.figure(figsize=(4.2, 3.5))
    plt.imshow(mat.detach().cpu())
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(title)
    plt.xlabel("input dim"); plt.ylabel("output dim")
    plt.show()


### Zoom-in: visual prompt tuning changes the image the frozen model sees

Visual prompt tuning adds a small learnable patch to the input image (typically at the corners).
The pretrained ViT is unchanged; only what it **sees** is different.

This is a direct vision analogue of soft prompt tuning in NLP.

In [ ]:
def draw_prompt_zoom(img_patches=8, prompt_patches=2):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.set_xlim(0, 13); ax.set_ylim(0, 6); ax.axis("off")
    ax.text(6.5, 5.55, "Visual prompt zoom-in: trainable pixels before a frozen ViT",
            ha="center", fontsize=14, fontweight="bold")
    for i in range(img_patches):
        add_box(ax, (0.6 + i*0.55, 3.6), 0.42, 0.55, f"p{i+1}", fc="#ffffff", fontsize=8)
    ax.text(2.6, 4.45, "image patches (original)", ha="center", fontsize=10)
    for i in range(prompt_patches):
        add_box(ax, (0.6 + i*0.55, 2.35), 0.42, 0.55, f"d{i+1}", fc="#f3f3f3", fontsize=8)
    ax.text(1.0, 1.95, "trainable prompt\n(added to corner)", ha="center", fontsize=10)
    add_arrow(ax, (4.9, 3.85), (5.8, 3.2), "x + prompt")
    for i in range(img_patches):
        fc = "#f3f3f3" if i < prompt_patches else "#ffffff"
        add_box(ax, (6.0 + i*0.45, 2.9), 0.35, 0.5, f"p{i+1}", fc=fc, fontsize=7)
    add_box(ax, (9.7, 2.55), 2.2, 1.2, "frozen\nViT backbone", fc="#ffffff", lw=2)
    add_arrow(ax, (9.15, 3.15), (9.7, 3.15))
    add_box(ax, (12.2, 2.75), 0.65, 0.8, "CLS")
    add_arrow(ax, (11.9, 3.15), (12.2, 3.15))
    ax.text(6.45, 1.1, "Only the prompt pixels are trained; the ViT checkpoint is unchanged.",
            ha="center", fontsize=11)
    plt.show()


draw_prompt_zoom()


In [ ]:
torch.manual_seed(4)
prompt = torch.randn(3, 8, 8) * 0.02

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
titles = ["Red channel", "Green channel", "Blue channel"]
for ax, ch, title in zip(axes, prompt, titles):
    im = ax.imshow(ch, cmap="RdBu", vmin=-0.06, vmax=0.06)
    ax.set_title(title); ax.axis("off")
plt.colorbar(im, ax=axes, fraction=0.03, pad=0.04)
plt.suptitle("Soft visual prompt: learnable pixel values added to corner patches", y=1.02)
plt.tight_layout()
plt.show()

print("Trainable prompt params:", prompt.numel())
print("Capacity grows with prompt area x channels.")


### Zoom-in: adapters add a residual bottleneck after the backbone

Adapters insert a small trainable branch in the hidden-state stream:
1. down-project to bottleneck
2. nonlinearity
3. up-project back
4. add as residual

In our setup the adapter is applied to the pooled CLS feature after the frozen ViT.

In [ ]:
def draw_adapter_zoom(d_model=12, bottleneck=4):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.set_xlim(0, 13); ax.set_ylim(0, 6); ax.axis("off")
    ax.text(6.5, 5.55, "Adapter zoom-in: a small residual branch in activation space",
            ha="center", fontsize=14, fontweight="bold")
    add_box(ax, (0.6, 2.7), 1.0, 0.8, "h")
    add_box(ax, (2.2, 2.45), 2.2, 1.3, "frozen\nViT backbone", fc="#ffffff", lw=2)
    add_arrow(ax, (1.6, 3.1), (2.2, 3.1))
    add_arrow(ax, (4.4, 3.1), (5.2, 3.1), "CLS features")
    add_arrow(ax, (1.1, 2.7), (3.1, 1.45), "copy h", rad=0.2)
    add_box(ax, (3.1, 0.85), 1.5, 0.8, f"down\n{d_model}->{bottleneck}", fc="#f3f3f3")
    add_box(ax, (5.0, 0.85), 1.2, 0.8, "GELU", fc="#f3f3f3")
    add_box(ax, (6.6, 0.85), 1.5, 0.8, f"up\n{bottleneck}->{d_model}", fc="#f3f3f3")
    add_arrow(ax, (4.6, 1.25), (5.0, 1.25))
    add_arrow(ax, (6.2, 1.25), (6.6, 1.25))
    add_arrow(ax, (8.1, 1.25), (8.8, 2.8), "residual dh")
    add_box(ax, (8.7, 2.65), 0.7, 0.9, "+")
    add_arrow(ax, (5.2, 3.1), (8.7, 3.1))
    add_box(ax, (10.3, 2.7), 1.2, 0.8, "h + dh")
    add_arrow(ax, (9.4, 3.1), (10.3, 3.1))
    params = d_model*bottleneck + bottleneck + bottleneck*d_model + d_model
    ax.text(6.5, 0.25, f"Trainable adapter params ~ 2 x d_model x bottleneck ~ {params} (incl. biases).",
            ha="center", fontsize=10)
    plt.show()


draw_adapter_zoom()


In [ ]:
torch.manual_seed(5)
d_model, bottleneck = 16, 4
W_down = torch.randn(bottleneck, d_model) * 0.1
W_up   = torch.randn(d_model, bottleneck) * 0.1

for title, mat in [("Adapter down projection", W_down), ("Adapter up projection", W_up)]:
    plt.figure(figsize=(4.6, 3.0))
    plt.imshow(mat)
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(title)
    plt.xlabel("input dim"); plt.ylabel("output dim")
    plt.show()

print("Adapter bottleneck params:", W_down.numel() + W_up.numel())
print("Full d_model x d_model matrix would have:", d_model * d_model)


### Zoom-in: BitFit trains only existing bias vectors

For a ViT backbone the trainable objects are:
- attention projection biases
- FFN biases (`linear1.bias`, `linear2.bias`)
- LayerNorm beta

The expressive power is limited, but the number of trainable parameters is tiny.

In [ ]:
def draw_bitfit_zoom():
    fig, ax = plt.subplots(figsize=(11.5, 4.6))
    ax.set_xlim(0, 12); ax.set_ylim(0, 5.5); ax.axis("off")
    ax.text(6, 5.15, "BitFit zoom-in: freeze weights, move only bias vectors",
            ha="center", fontsize=14, fontweight="bold")
    add_box(ax, (0.7, 2.4), 1.0, 0.8, "x")
    add_box(ax, (2.4, 1.6), 2.4, 2.3, "frozen weight W\nlarge matrix", fc="#ffffff", lw=2)
    add_arrow(ax, (1.7, 2.8), (2.4, 2.8))
    add_box(ax, (5.7, 3.0), 1.9, 0.55, "trainable bias b", fc="#f3f3f3")
    add_arrow(ax, (4.8, 2.8), (5.3, 2.8))
    add_box(ax, (5.3, 2.35), 0.6, 0.9, "+")
    add_arrow(ax, (6.65, 3.0), (5.9, 3.0), "add")
    add_arrow(ax, (5.9, 2.8), (8.1, 2.8))
    add_box(ax, (8.1, 2.4), 1.2, 0.8, "y")
    ax.text(6, 0.8, "The expressive power is limited, but the number of trainable parameters is tiny.",
            ha="center", fontsize=11)
    plt.show()


draw_bitfit_zoom()


In [ ]:
torch.manual_seed(6)
W = torch.randn(16, 16); b = torch.randn(16) * 0.05

plt.figure(figsize=(4, 3.2))
plt.imshow(W); plt.colorbar(fraction=0.046, pad=0.04)
plt.title("Frozen weight matrix W")
plt.xlabel("input dim"); plt.ylabel("output dim")
plt.show()

plt.figure(figsize=(6, 1.5))
plt.imshow(b.view(1, -1)); plt.colorbar(fraction=0.046, pad=0.04)
plt.title("Trainable BitFit bias vector b")
plt.yticks([]); plt.xlabel("output dim")
plt.show()

print("Trainable BitFit params:", b.numel(), "vs full matrix:", W.numel())


### Zoom-in: linear probing trains only the readout

> Are the pretrained representations already linearly separable enough for my task?

If yes, linear probing can be surprisingly strong.
If no, it plateaus quickly because it cannot reshape the internal representation.

In [ ]:
def draw_linear_probe_zoom():
    fig, ax = plt.subplots(figsize=(11.5, 4.5))
    ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis("off")
    ax.text(6, 4.65, "Linear probing: frozen feature extractor, trainable readout",
            ha="center", fontsize=14, fontweight="bold")
    add_box(ax, (0.7, 2.0), 1.2, 0.8, "input")
    add_box(ax, (2.6, 1.55), 3.0, 1.7, "frozen pretrained\nViT backbone", fc="#ffffff", lw=2)
    add_box(ax, (6.5, 2.0), 1.4, 0.8, "CLS features h")
    add_box(ax, (8.8, 1.7), 1.9, 1.4, "trainable\nlinear head", fc="#f3f3f3")
    add_box(ax, (11.0, 2.0), 0.8, 0.8, "y_hat")
    add_arrow(ax, (1.9, 2.4), (2.6, 2.4))
    add_arrow(ax, (5.6, 2.4), (6.5, 2.4))
    add_arrow(ax, (7.9, 2.4), (8.8, 2.4))
    add_arrow(ax, (10.7, 2.4), (11.0, 2.4))
    ax.text(6, 0.65, "Good first question: is a cheap linear boundary enough?",
            ha="center", fontsize=11)
    plt.show()


draw_linear_probe_zoom()


### Zoom-in: partial and full fine-tuning move the checkpoint itself

In [ ]:
def draw_finetuning_zoom():
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.set_xlim(0, 12); ax.set_ylim(0, 6); ax.axis("off")
    ax.text(6, 5.6, "Fine-tuning reference: which pretrained blocks are allowed to move?",
            ha="center", fontsize=14, fontweight="bold")
    ax.text(3, 4.8, "Partial fine-tuning", ha="center", fontsize=12, fontweight="bold")
    for i in range(5):
        fc = "#f3f3f3" if i >= 3 else "#ffffff"
        label = f"block {i+1}\n" + ("train" if i >= 3 else "frozen")
        add_box(ax, (0.7 + i*0.95, 3.45), 0.8, 0.75, label, fc=fc, fontsize=8)
    add_box(ax, (5.8, 3.45), 0.9, 0.75, "head\ntrain", fc="#f3f3f3", fontsize=8)
    ax.text(3, 2.45, "Full fine-tuning", ha="center", fontsize=12, fontweight="bold")
    for i in range(5):
        add_box(ax, (0.7 + i*0.95, 1.1), 0.8, 0.75, f"block {i+1}\ntrain", fc="#f3f3f3", fontsize=8)
    add_box(ax, (5.8, 1.1), 0.9, 0.75, "head\ntrain", fc="#f3f3f3", fontsize=8)
    add_box(ax, (7.3, 3.25), 3.8, 1.2, "More trainable parameters\nmore capacity\nmore memory/storage", fc="#ffffff")
    add_box(ax, (7.3, 1.0), 3.8, 1.2, "Strong reference baseline\nbut task-specific checkpoints\ncan be large", fc="#ffffff")
    plt.show()


draw_finetuning_zoom()


### One-page visual summary

In [ ]:
summary = pd.DataFrame([
    {"method": "linear probing",       "trainable state": "head",                     "acts on": "label/readout space",         "main knob": "none / head size"},
    {"method": "visual prompt tuning", "trainable state": "prompt pixels/patches",    "acts on": "input image space",           "main knob": "prompt size"},
    {"method": "adapters",             "trainable state": "bottleneck modules",        "acts on": "hidden activation path",      "main knob": "bottleneck width"},
    {"method": "LoRA",                 "trainable state": "A and B matrices",          "acts on": "effective weight space",      "main knob": "rank r"},
    {"method": "BitFit",               "trainable state": "biases only",               "acts on": "small existing param subset", "main knob": "which layers"},
    {"method": "partial fine-tuning",  "trainable state": "selected pretrained layers","acts on": "checkpoint weights (partial)","main knob": "which blocks"},
    {"method": "full fine-tuning",     "trainable state": "all weights",               "acts on": "entire checkpoint",           "main knob": "lr / regularisation"},
])
summary


## 6) A small synthetic training exercise

Labels are derived from the frozen backbone's output so all methods are in principle able to solve the task.

In [ ]:
def make_synthetic_dataset(n=256, image_size=32, n_classes=3):
    x = torch.randn(n, 3, image_size, image_size)
    with torch.no_grad():
        h = backbone(x.to(device)).cpu()
    W = torch.randn(backbone.feature_dim, n_classes)
    y = (h @ W).argmax(dim=-1)
    return x, y


X, y = make_synthetic_dataset()
X_train, y_train = X[:200].to(device), y[:200].to(device)
X_val,   y_val   = X[200:].to(device), y[200:].to(device)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)


In [ ]:
def build_method(name):
    bb = copy.deepcopy(backbone).to(device)
    D, K = backbone.feature_dim, N_CLASSES
    if name == "linear_probe":  return LinearProbeModel(bb, D, K)
    if name == "visual_prompt": return PromptTunedClassifier(bb, D, K, prompt_size=8)
    if name == "adapter":       return AdapterHeadClassifier(bb, D, K, bottleneck_dim=32)
    if name == "lora":          return LoRAClassifier(bb, D, K, target_modules=["linear1", "linear2"], rank=8)
    if name == "bitfit":        return BitFitClassifier(bb, D, K)
    if name == "partial_ft":
        last = list(bb.transformer.layers)[-1]
        return PartialFineTuneClassifier(bb, D, K, modules_to_unfreeze=[last])
    raise ValueError(name)


def fit(name, epochs=30, lr=3e-3):
    model = build_method(name).to(device)
    opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    rows  = []
    for epoch in range(epochs):
        model.train()
        logits = model(X_train)
        loss   = F.cross_entropy(logits, y_train)
        opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            train_acc  = (model(X_train).argmax(-1) == y_train).float().mean().item()
            val_logits = model(X_val)
            val_loss   = F.cross_entropy(val_logits, y_val).item()
            val_acc    = (val_logits.argmax(-1) == y_val).float().mean().item()
        rows.append({"epoch": epoch+1, "method": name,
                     "train_acc": train_acc, "val_acc": val_acc,
                     "train_loss": float(loss), "val_loss": val_loss,
                     "trainable_params": count_trainable_parameters(model)})
    return pd.DataFrame(rows)


methods = ["linear_probe", "visual_prompt", "adapter", "lora", "bitfit", "partial_ft"]
runs = [fit(m) for m in methods]
hist = pd.concat(runs, ignore_index=True)
hist.tail()


In [ ]:
plt.figure(figsize=(8, 4))
for method, g in hist.groupby("method"):
    plt.plot(g["epoch"], g["val_acc"], label=method)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Synthetic task: same frozen TinyViT, different adaptation strategies")
plt.legend()
plt.tight_layout()
plt.show()


## 7) Practical takeaway

| Method | Trainable object | Where it acts | Mental model |
|---|---|---|---|
| Linear probing | classifier head | output / label space | reuse features as-is |
| Visual prompt | prompt pixels | input image space | steer the frozen model by changing what it sees |
| Adapter | bottleneck residual | hidden-state path | add a small specialist module |
| LoRA | low-rank weight delta | weight space | change how a layer computes, cheaply |
| BitFit | biases only | parameter subset | nudge pretrained behavior |
| Partial fine-tuning | last blocks | checkpoint weights | let the top layers specialise |

### Rule-of-thumb hypotheses to test in later notebooks

- **Linear probing** is the first baseline when you think the pretrained features are already close to your task.
- **Visual prompt tuning** is attractive when you want many task-specific states while keeping the base weights fixed.
- **Adapters** are useful when you want modularity and clear insertion points.
- **LoRA** is a strong default when you want more expressivity than prompts or BitFit, but far fewer trainable weights than full fine-tuning.
- **BitFit** is a tiny, cheap baseline for small or mild domain shifts.
- **Partial fine-tuning** is a good middle-ground when the top layers need to specialise substantially.